# 01 — RoBERTa Fine-tune: CVE → CWE Top 25 Classification

**Goal:** Fine-tune `roberta-base` on the `xamxte/cve-to-cwe` dataset, filtered to MITRE Top 25 CWEs.  
**Primary metric:** Macro F1 (not accuracy — class imbalance makes accuracy misleading).  
**Output:** Trained checkpoint pushed to HF Hub at `your-username/vuln-classifier-roberta`.

Expected runtime on Kaggle T4: ~45–60 min for 3 epochs.

## 0. Install dependencies

In [2]:
%%capture
!pip install transformers datasets evaluate scikit-learn huggingface_hub accelerate -q

## 1. Config — change only this cell

In [ ]:
# ── User config ──────────────────────────────────────────────────────────────
HF_USERNAME     = ""          # your Hugging Face username
HF_REPO_NAME    = "vuln-classifier-roberta"
HF_TOKEN        = ""                # paste your HF write token here
                                           # or use: from kaggle_secrets import UserSecretsClient

BASE_MODEL      = "roberta-base"
DATASET_ID      = "xamxte/cve-to-cwe"

MAX_LENGTH      = 256    # CVE descriptions are short; 256 is plenty
BATCH_SIZE      = 16     # safe for T4 15GB
EPOCHS          = 3
LR              = 2e-5
WARMUP_RATIO    = 0.1
SEED            = 42

# ── Label space ──────────────────────────────────────────────────────────────
MITRE_TOP_25 = [
    "CWE-787", "CWE-79",  "CWE-89",  "CWE-416", "CWE-78",
    "CWE-20",  "CWE-125", "CWE-22",  "CWE-352", "CWE-434",
    "CWE-862", "CWE-476", "CWE-287", "CWE-190", "CWE-502",
    "CWE-77",  "CWE-119", "CWE-798", "CWE-918", "CWE-306",
    "CWE-362", "CWE-269", "CWE-94",  "CWE-863", "CWE-276"
]

PRIMARY_METRIC  = "macro_f1"

## 2. HF Hub login

In [4]:
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("Logged in to HF Hub")

Logged in to HF Hub


## 3. Load and filter dataset

In [5]:
from datasets import load_dataset

raw = load_dataset(DATASET_ID)
print("Raw splits:", raw)
print("Columns:", raw["train"].column_names)
print("Sample row:", raw["train"][0])

Raw splits: DatasetDict({
    train: Dataset({
        features: ['cve_id', 'description', 'cwe_id', 'label', 'attack_techniques'],
        num_rows: 234770
    })
    validation: Dataset({
        features: ['cve_id', 'description', 'cwe_id', 'label', 'attack_techniques'],
        num_rows: 27896
    })
    test: Dataset({
        features: ['cve_id', 'description', 'cwe_id', 'label', 'attack_techniques'],
        num_rows: 27780
    })
})
Columns: ['cve_id', 'description', 'cwe_id', 'label', 'attack_techniques']
Sample row: {'cve_id': 'CVE-2025-7782', 'description': "The WP JobHunt plugin for WordPress, used by the JobCareer theme, is vulnerable to unauthorized modification of data due to a missing capability check on the 'cs_update_application_status_callback' function in all versions up to, and including, 7.7. This makes it possible for authenticated attackers, with Candidate-level access and above, to inject cross-site scripting into the 'status' parameter of applied jobs for any 

In [6]:
# Filter to Top 25 only
# Column names may vary — inspect above and adjust 'cwe_id' and 'description' if needed
TEXT_COL  = "description"   # the CVE/vuln description text
LABEL_COL = "cwe_id"        # the CWE label

def keep_top25(example):
    return example[LABEL_COL] in MITRE_TOP_25

filtered = raw.filter(keep_top25, num_proc=2)
print("After Top 25 filter:", filtered)

# Class distribution — check for severe imbalance
from collections import Counter
dist = Counter(filtered["train"][LABEL_COL])
for cwe, count in sorted(dist.items(), key=lambda x: -x[1]):
    print(f"  {cwe}: {count:,}")

After Top 25 filter: DatasetDict({
    train: Dataset({
        features: ['cve_id', 'description', 'cwe_id', 'label', 'attack_techniques'],
        num_rows: 145050
    })
    validation: Dataset({
        features: ['cve_id', 'description', 'cwe_id', 'label', 'attack_techniques'],
        num_rows: 20379
    })
    test: Dataset({
        features: ['cve_id', 'description', 'cwe_id', 'label', 'attack_techniques'],
        num_rows: 20279
    })
})
  CWE-79: 33,858
  CWE-89: 15,619
  CWE-22: 8,047
  CWE-862: 7,533
  CWE-78: 7,132
  CWE-125: 6,770
  CWE-787: 6,508
  CWE-20: 6,299
  CWE-352: 6,270
  CWE-416: 6,009
  CWE-119: 5,943
  CWE-476: 4,931
  CWE-434: 3,697
  CWE-306: 3,313
  CWE-190: 3,210
  CWE-863: 3,078
  CWE-287: 2,871
  CWE-269: 2,601
  CWE-94: 2,481
  CWE-362: 2,278
  CWE-502: 2,109
  CWE-918: 1,640
  CWE-276: 1,337
  CWE-798: 1,271
  CWE-77: 245


In [7]:
# Build label → int mapping (deterministic sort so it's reproducible)
label2id = {cwe: i for i, cwe in enumerate(sorted(MITRE_TOP_25))}
id2label = {i: cwe for cwe, i in label2id.items()}
NUM_LABELS = len(label2id)
print(f"Label space: {NUM_LABELS} classes")
print(label2id)

Label space: 25 classes
{'CWE-119': 0, 'CWE-125': 1, 'CWE-190': 2, 'CWE-20': 3, 'CWE-22': 4, 'CWE-269': 5, 'CWE-276': 6, 'CWE-287': 7, 'CWE-306': 8, 'CWE-352': 9, 'CWE-362': 10, 'CWE-416': 11, 'CWE-434': 12, 'CWE-476': 13, 'CWE-502': 14, 'CWE-77': 15, 'CWE-78': 16, 'CWE-787': 17, 'CWE-79': 18, 'CWE-798': 19, 'CWE-862': 20, 'CWE-863': 21, 'CWE-89': 22, 'CWE-918': 23, 'CWE-94': 24}


## 4. Tokenize

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    enc = tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    enc["labels"] = [label2id[cwe] for cwe in batch[LABEL_COL]]
    return enc

tokenized = filtered.map(tokenize, batched=True, num_proc=2,
                         remove_columns=filtered["train"].column_names)
tokenized.set_format("torch")
print("Tokenized:", tokenized)

Tokenized: DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 145050
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 20379
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 20279
    })
})


## 5. Model

In [9]:
from transformers import AutoModelForSequenceClassification
import torch

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
model = model.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda


## 6. Metrics — macro F1 as primary

In [10]:
import numpy as np
from sklearn.metrics import f1_score, classification_report

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    accuracy  = (preds == labels).mean()

    # Per-class F1 — logged so you can see weak spots
    per_class = f1_score(labels, preds, average=None, zero_division=0)
    per_class_dict = {f"f1_{id2label[i]}": round(float(v), 4)
                      for i, v in enumerate(per_class)}

    return {
        "macro_f1": round(macro_f1, 4),
        "accuracy": round(float(accuracy), 4),
        **per_class_dict
    }

## 7. Training

In [11]:
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir=f"/kaggle/working/{HF_REPO_NAME}",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model=PRIMARY_METRIC,
    greater_is_better=True,
    logging_steps=50,
    fp16=True,                  # T4 supports fp16 — cuts memory ~40%
    seed=SEED,
    report_to="none"            # no wandb needed
)

# Split: use dataset's own test split if available, else carve 10% from train
if "test" in tokenized:
    train_ds, eval_ds = tokenized["train"], tokenized["test"]
else:
    split = tokenized["train"].train_test_split(test_size=0.1, seed=SEED)
    train_ds, eval_ds = split["train"], split["test"]

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer)
)

print(f"Training on {len(train_ds):,} samples, evaluating on {len(eval_ds):,}")
trainer.train()

Training on 145,050 samples, evaluating on 20,279


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy,F1 Cwe-119,F1 Cwe-125,F1 Cwe-190,F1 Cwe-20,F1 Cwe-22,F1 Cwe-269,F1 Cwe-276,F1 Cwe-287,F1 Cwe-306,F1 Cwe-352,F1 Cwe-362,F1 Cwe-416,F1 Cwe-434,F1 Cwe-476,F1 Cwe-502,F1 Cwe-77,F1 Cwe-78,F1 Cwe-787,F1 Cwe-79,F1 Cwe-798,F1 Cwe-862,F1 Cwe-863,F1 Cwe-89,F1 Cwe-918,F1 Cwe-94
1,0.589352,0.461629,0.830100,0.930600,0.833200,0.935400,0.922900,0.786000,0.972500,0.681900,0.745600,0.802100,0.694900,0.993800,0.916700,0.921900,0.945000,0.942800,0.924300,0.000000,0.882000,0.848100,0.992100,0.953000,0.895300,0.518700,0.996400,0.963000,0.685100
2,0.463213,0.405284,0.847200,0.940200,0.864000,0.945800,0.943800,0.836100,0.975200,0.700200,0.801700,0.828500,0.725400,0.994700,0.901900,0.930700,0.948800,0.946300,0.946000,0.000000,0.901400,0.886000,0.994100,0.953300,0.897600,0.562800,0.996400,0.963100,0.737200
3,0.378637,0.401293,0.850100,0.942100,0.858700,0.950300,0.939900,0.837100,0.973800,0.699200,0.785700,0.838300,0.750600,0.995900,0.902300,0.934900,0.955800,0.946000,0.933500,0.000000,0.912700,0.882400,0.994900,0.950800,0.907800,0.603700,0.997500,0.961600,0.740400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=13599, training_loss=0.6055739971020352, metrics={'train_runtime': 11896.9309, 'train_samples_per_second': 36.577, 'train_steps_per_second': 1.143, 'total_flos': 5.72582096909568e+16, 'train_loss': 0.6055739971020352, 'epoch': 3.0})

## 8. Final eval — full classification report

In [12]:
results = trainer.evaluate()
print(f"\n{'='*50}")
print(f"Macro F1 (primary): {results['eval_macro_f1']}")
print(f"Accuracy:           {results['eval_accuracy']}")
print(f"{'='*50}")

# Full per-class breakdown — paste this into your README
preds_out = trainer.predict(eval_ds)
preds = np.argmax(preds_out.predictions, axis=-1)
labels = preds_out.label_ids

print("\nPer-class F1 (sorted by score):")
report = classification_report(
    labels, preds,
    target_names=[id2label[i] for i in range(NUM_LABELS)],
    zero_division=0
)
print(report)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Macro F1 (primary): 0.8501
Accuracy:           0.9421


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Per-class F1 (sorted by score):
              precision    recall  f1-score   support

     CWE-119       0.87      0.85      0.86       762
     CWE-125       0.95      0.95      0.95      1028
     CWE-190       0.97      0.91      0.94       376
      CWE-20       0.83      0.84      0.84       704
      CWE-22       0.97      0.98      0.97      1082
     CWE-269       0.68      0.72      0.70       238
     CWE-276       0.81      0.76      0.79       130
     CWE-287       0.87      0.81      0.84       314
     CWE-306       0.67      0.85      0.75       200
     CWE-352       1.00      0.99      1.00      1221
     CWE-362       0.90      0.90      0.90       241
     CWE-416       0.92      0.95      0.93       878
     CWE-434       0.95      0.97      0.96       447
     CWE-476       0.94      0.95      0.95       626
     CWE-502       0.92      0.94      0.93       313
      CWE-77       0.00      0.00      0.00        48
      CWE-78       0.89      0.94      0.91     

## 9. Save label map + push to HF Hub

In [13]:
import json, os

# Save label map alongside model — pipeline/classifier.py needs this at runtime
label_map = {"label2id": label2id, "id2label": id2label}
map_path = f"/kaggle/working/{HF_REPO_NAME}/label_map.json"
with open(map_path, "w") as f:
    json.dump(label_map, f, indent=2)
print("Saved label_map.json")

# Push model + tokenizer + label map to HF Hub
HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"
trainer.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)

# Upload label map as a separate file
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj=map_path,
    path_in_repo="label_map.json",
    repo_id=HF_REPO_ID,
    token=HF_TOKEN
)

print(f"\nDone. Model live at: https://huggingface.co/{HF_REPO_ID}")

Saved label_map.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.



Done. Model live at: https://huggingface.co/martynattakit/vuln-classifier-roberta


## 10. Quick sanity check — test a few CVE descriptions

In [14]:
from transformers import pipeline as hf_pipeline

clf = hf_pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
    top_k=3
)

test_cases = [
    # Expected: CWE-89 (SQL injection)
    "The login form passes user input directly into the SQL query without parameterization, "
    "allowing an attacker to inject arbitrary SQL commands.",

    # Expected: CWE-79 (XSS)
    "User-supplied data is reflected in the HTTP response without encoding, "
    "enabling cross-site scripting attacks.",

    # Expected: CWE-787 (out-of-bounds write)
    "A heap buffer overflow in the image parsing library allows an attacker to write "
    "beyond the allocated buffer by supplying a crafted PNG file."
]

for text in test_cases:
    results = clf(text)
    print(f"Input: {text[:80]}...")
    for r in results[0]:
        print(f"  {r['label']}: {r['score']:.3f}")
    print()

Input: The login form passes user input directly into the SQL query without parameteriz...
  CWE-89: 1.000
  CWE-79: 0.000
  CWE-94: 0.000

Input: User-supplied data is reflected in the HTTP response without encoding, enabling ...
  CWE-79: 0.999
  CWE-434: 0.000
  CWE-94: 0.000

Input: A heap buffer overflow in the image parsing library allows an attacker to write ...
  CWE-787: 0.997
  CWE-119: 0.001
  CWE-416: 0.000



---
## What to record after this run

Paste these into your README under **Model Performance**:

| Metric | Value |
|--------|-------|
| Macro F1 | _(fill in)_ |
| Accuracy | _(fill in)_ |
| Weakest CWE class | _(fill in from per-class report)_ |
| Training time | _(fill in)_ |
| HF Hub URL | `https://huggingface.co/your-username/vuln-classifier-roberta` |

## Next: `02_qwen_qlora.ipynb`

Only start notebook 02 once this one pushes a checkpoint to HF Hub and the sanity check outputs look sane.  
If any Top 25 CWE has per-class F1 < 0.30, note it — that class either has too few samples or the descriptions are too ambiguous. Don't try to fix it now; flag it in the README and move on.